In [3]:
from IPython.display import HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>Real-Time 3D Surface Motion Model</title>
<style>
  :root {
    --bg: #050b16;
    --panel: rgba(5, 14, 28, 0.78);
    --grid: rgba(120, 190, 255, 0.20);
    --grid-strong: rgba(180, 230, 255, 0.62);
    --text: rgba(246, 250, 255, 0.96);
    --muted: rgba(210, 230, 255, 0.74);
  }

  * { box-sizing: border-box; }

  body {
    margin: 0;
    min-height: 100vh;
    background:
      radial-gradient(circle at 45% 8%, rgba(54, 142, 255, .22), transparent 34%),
      radial-gradient(circle at 82% 44%, rgba(255, 221, 40, .08), transparent 24%),
      linear-gradient(145deg, #020610 0%, #071123 58%, #010308 100%);
    font-family: Inter, Segoe UI, Roboto, Arial, sans-serif;
    color: var(--text);
    overflow: hidden;
  }

  .wrap {
    height: 100vh;
    display: grid;
    grid-template-rows: auto 1fr auto;
    gap: 12px;
    padding: 18px;
  }

  header {
    text-align: center;
  }

  h1 {
    margin: 0;
    font-size: clamp(24px, 4vw, 42px);
    font-weight: 850;
    letter-spacing: .035em;
    text-shadow: 0 0 18px rgba(255,255,255,.42), 0 0 40px rgba(91,172,255,.28);
  }

  .subtitle {
    margin-top: 5px;
    font-size: 13px;
    color: var(--muted);
  }

  .stage {
    position: relative;
    border: 1px solid rgba(160, 220, 255, .22);
    border-radius: 22px;
    overflow: hidden;
    background:
      linear-gradient(180deg, rgba(255,255,255,.045), rgba(255,255,255,.012)),
      rgba(2, 9, 21, .62);
    box-shadow:
      0 24px 78px rgba(0,0,0,.48),
      inset 0 0 38px rgba(65,151,255,.10);
  }

  canvas {
    width: 100%;
    height: 100%;
    display: block;
  }

  .hud {
    position: absolute;
    top: 18px;
    right: 18px;
    width: 240px;
    padding: 12px 14px;
    border-radius: 16px;
    background: var(--panel);
    border: 1px solid rgba(180, 220, 255, .27);
    box-shadow: 0 0 24px rgba(70, 170, 255, .14);
    backdrop-filter: blur(8px);
    color: var(--muted);
    font-size: 13px;
    line-height: 1.48;
  }

  .hud strong {
    color: var(--text);
  }

  .controls {
    display: flex;
    gap: 12px;
    align-items: center;
    justify-content: center;
    flex-wrap: wrap;
  }

  button {
    border: 1px solid rgba(180,220,255,.30);
    color: var(--text);
    background: rgba(12, 30, 55, .86);
    border-radius: 999px;
    padding: 10px 16px;
    font-weight: 760;
    cursor: pointer;
    box-shadow: 0 0 20px rgba(46, 135, 255, .12);
  }

  button:hover { background: rgba(18, 45, 80, .96); }

  label {
    color: var(--muted);
    font-weight: 650;
    display: inline-flex;
    align-items: center;
    gap: 8px;
  }

  input[type="range"] { accent-color: #59b6ff; }
</style>
</head>
<body>
<div class="wrap">
  <header>
    <h1>demo</h1>
    <div class="subtitle">Real-time 3D motion simulation of stacked amplitude surfaces</div>
  </header>

  <main class="stage">
    <canvas id="sim"></canvas>
    <div class="hud">
      <div><strong>Live model state</strong></div>
      <div id="frameReadout">Frame: 0</div>
      <div id="waveReadout">Surface phase: 0.00</div>
      <div id="sampleReadout">Current sample: 0.0/hr</div>
      <div id="ampReadout">Amplitude: --</div>
    </div>
  </main>

  <section class="controls">
    <button id="pause">Pause</button>
    <button id="reset">Reset</button>
    <label>Speed <input id="speed" type="range" min="0.2" max="3.0" step="0.1" value="1"></label>
    <label>Ripple <input id="ripple" type="range" min="0.2" max="2.4" step="0.1" value="1.2"></label>
    <label>Glow <input id="glow" type="range" min="2" max="22" step="1" value="11"></label>
  </section>
</div>

<script>
const canvas = document.getElementById("sim");
const ctx = canvas.getContext("2d");

const pauseBtn = document.getElementById("pause");
const resetBtn = document.getElementById("reset");
const speedSlider = document.getElementById("speed");
const rippleSlider = document.getElementById("ripple");
const glowSlider = document.getElementById("glow");

const frameReadout = document.getElementById("frameReadout");
const waveReadout = document.getElementById("waveReadout");
const sampleReadout = document.getElementById("sampleReadout");
const ampReadout = document.getElementById("ampReadout");

let W, H, dpr;
let running = true;
let start = performance.now();
let phaseHold = 0;
let frame = 0;

function resize() {
  dpr = Math.max(1, Math.min(2, window.devicePixelRatio || 1));
  W = canvas.clientWidth;
  H = canvas.clientHeight;
  canvas.width = Math.floor(W * dpr);
  canvas.height = Math.floor(H * dpr);
  ctx.setTransform(dpr, 0, 0, dpr, 0, 0);
}
window.addEventListener("resize", resize);
resize();

const C = {
  text: "rgba(246,250,255,.96)",
  muted: "rgba(215,235,255,.74)",
  grid: "rgba(120,190,255,.20)",
  gridStrong: "rgba(190,230,255,.58)",
  cyan: "#49c8ff",
  purple: "#49118b",
  blue: "#1557d4",
  teal: "#12b6b6",
  green: "#52de58",
  yellow: "#fff042"
};

function clamp(v, a, b) { return Math.max(a, Math.min(b, v)); }

function lerp(a, b, t) { return a + (b - a) * t; }

function hexToRgb(hex) {
  const n = parseInt(hex.slice(1), 16);
  return [(n >> 16) & 255, (n >> 8) & 255, n & 255];
}

function mixColor(a, b, t) {
  const A = hexToRgb(a), B = hexToRgb(b);
  return `rgb(${Math.round(lerp(A[0],B[0],t))},${Math.round(lerp(A[1],B[1],t))},${Math.round(lerp(A[2],B[2],t))})`;
}

function viridisLike(t) {
  t = clamp(t, 0, 1);
  const stops = [
    [0.00, C.purple],
    [0.26, C.blue],
    [0.52, C.teal],
    [0.76, C.green],
    [1.00, C.yellow]
  ];
  for (let i = 0; i < stops.length - 1; i++) {
    const [p0, c0] = stops[i];
    const [p1, c1] = stops[i+1];
    if (t >= p0 && t <= p1) return mixColor(c0, c1, (t - p0) / (p1 - p0));
  }
  return C.yellow;
}

function text(str, x, y, size=13, align="center", rot=0, weight=750) {
  ctx.save();
  ctx.translate(x, y);
  ctx.rotate(rot);
  ctx.font = `${weight} ${size}px Inter, Segoe UI, Arial, sans-serif`;
  ctx.textAlign = "center";
  ctx.textBaseline = "middle";
  ctx.fillStyle = C.text;
  ctx.shadowColor = "rgba(145,215,255,.50)";
  ctx.shadowBlur = 8;
  ctx.fillText(str, 0, 0);
  ctx.restore();
}

function project(panel, p) {
  // p.x: 0..50, p.y: 0..24, p.z: zmin..zmax
  const {cx, cy, sx, sy, sz, zmin, zmax} = panel;
  const nx = p.x / 50;
  const ny = p.y / 24;
  const nz = (p.z - zmin) / (zmax - zmin);

  // Perspective/isometric projection
  const px = cx + (nx - 0.5) * sx + (ny - 0.5) * sy;
  const py = cy + (nx - 0.5) * sx * 0.34 - (ny - 0.5) * sy * 0.30 - (nz - 0.5) * sz;
  return {x: px, y: py};
}

function ampTop(x, y, phase, ripple) {
  const base = 0.48 + 0.16 * (1 - x / 50) * 0.68 + 0.16 * (y / 24) * 0.32;
  const waves =
    0.0065 * ripple * Math.sin(x * 0.68 + phase * 2.1) +
    0.0045 * ripple * Math.cos(y * 0.92 - phase * 1.7) +
    0.0030 * ripple * Math.sin((x + y) * 0.33 + phase * 2.8);
  return clamp(base + waves, 0.48, 0.64);
}

function ampBottom(x, y, phase, ripple) {
  const base = 0.24 + 0.08 * (1 - x / 50) * 0.68 + 0.08 * (y / 24) * 0.32;
  const waves =
    0.0035 * ripple * Math.sin(x * 0.72 + phase * 2.3) +
    0.0025 * ripple * Math.cos(y * 0.95 - phase * 1.4) +
    0.0018 * ripple * Math.sin((x + y) * 0.36 + phase * 2.6);
  return clamp(base + waves, 0.24, 0.32);
}

function drawGrid(panel) {
  ctx.save();
  ctx.lineWidth = 1;

  for (let x = 0; x <= 50; x += 10) {
    const a = project(panel, {x, y:0, z:panel.zmin});
    const b = project(panel, {x, y:24, z:panel.zmin});
    const c = project(panel, {x, y:24, z:panel.zmax});
    ctx.strokeStyle = x % 20 === 0 ? C.gridStrong : C.grid;
    ctx.beginPath(); ctx.moveTo(a.x, a.y); ctx.lineTo(b.x, b.y); ctx.stroke();
    ctx.beginPath(); ctx.moveTo(b.x, b.y); ctx.lineTo(c.x, c.y); ctx.stroke();
  }

  for (let y = 0; y <= 24; y += 4) {
    const a = project(panel, {x:0, y, z:panel.zmin});
    const b = project(panel, {x:50, y, z:panel.zmin});
    ctx.strokeStyle = y % 8 === 0 ? C.gridStrong : C.grid;
    ctx.beginPath(); ctx.moveTo(a.x, a.y); ctx.lineTo(b.x, b.y); ctx.stroke();
  }

  const zStep = (panel.zmax - panel.zmin) / 8;
  for (let i = 0; i <= 8; i++) {
    const z = panel.zmin + zStep * i;
    const a = project(panel, {x:50, y:24, z});
    const b = project(panel, {x:0, y:24, z});
    ctx.strokeStyle = C.grid;
    ctx.beginPath(); ctx.moveTo(a.x, a.y); ctx.lineTo(b.x, b.y); ctx.stroke();
  }

  // Box edges
  const edges = [
    [{x:0,y:0,z:panel.zmin},{x:50,y:0,z:panel.zmin}],
    [{x:0,y:0,z:panel.zmin},{x:0,y:24,z:panel.zmin}],
    [{x:50,y:0,z:panel.zmin},{x:50,y:24,z:panel.zmin}],
    [{x:0,y:24,z:panel.zmin},{x:50,y:24,z:panel.zmin}],
    [{x:50,y:24,z:panel.zmin},{x:50,y:24,z:panel.zmax}],
    [{x:0,y:24,z:panel.zmax},{x:50,y:24,z:panel.zmax}],
    [{x:0,y:0,z:panel.zmin},{x:0,y:24,z:panel.zmax}],
  ];
  ctx.strokeStyle = "rgba(180,235,255,.80)";
  ctx.shadowColor = "rgba(70,190,255,.80)";
  ctx.shadowBlur = 9;
  ctx.lineWidth = 1.5;
  for (const [p0, p1] of edges) {
    const a = project(panel, p0), b = project(panel, p1);
    ctx.beginPath(); ctx.moveTo(a.x, a.y); ctx.lineTo(b.x, b.y); ctx.stroke();
  }

  ctx.restore();
}

function drawAxes(panel) {
  const base0 = project(panel, {x:0,y:0,z:panel.zmin});
  const xLab = project(panel, {x:28,y:0,z:panel.zmin});
  const yLab = project(panel, {x:42,y:18,z:panel.zmin});
  const zLab = project(panel, {x:50,y:24,z:(panel.zmin + panel.zmax) / 2});

  text("demo", panel.cx, panel.cy - panel.sz * .82, 26, "center", 0, 850);
  text("Sample Points Per Hour", xLab.x - 30, xLab.y + 46, 16, "center", 0.42, 760);
  text("Time (Hours)", yLab.x + 30, yLab.y + 42, 16, "center", -0.10, 760);
  text("Amplitude", zLab.x + 40, zLab.y, 14, "center", -Math.PI/2, 760);

  [0,10,20,30,40,50].forEach(x => {
    const p = project(panel, {x, y:0, z:panel.zmin});
    text(String(x), p.x, p.y + 19, 12);
  });

  [0,5,10,15,20].forEach(y => {
    const p = project(panel, {x:50, y, z:panel.zmin});
    text(String(y), p.x + 12, p.y + 13, 12);
  });

  const ticks = panel.zTicks;
  ticks.forEach(z => {
    const p = project(panel, {x:50,y:24,z});
    text(z.toFixed(2), p.x + 30, p.y, 11, "left");
  });
}

function drawSurface(panel, ampFn, phase, ripple, glow) {
  const rows = [];
  const stepX = 2;
  const stepY = 1.2;

  for (let y = 0; y <= 24.001; y += stepY) {
    const row = [];
    for (let x = 0; x <= 50.001; x += stepX) {
      row.push({x, y, z: ampFn(x, y, phase, ripple)});
    }
    rows.push(row);
  }

  // Draw cells back-to-front by y then x.
  ctx.save();
  for (let j = rows.length - 2; j >= 0; j--) {
    for (let i = 0; i < rows[j].length - 1; i++) {
      const p00 = rows[j][i];
      const p10 = rows[j][i+1];
      const p11 = rows[j+1][i+1];
      const p01 = rows[j+1][i];

      const avg = (p00.z + p10.z + p11.z + p01.z) / 4;
      const t = (avg - panel.zmin) / (panel.zmax - panel.zmin);
      const a = project(panel, p00), b = project(panel, p10), c = project(panel, p11), d = project(panel, p01);

      ctx.fillStyle = viridisLike(t);
      ctx.globalAlpha = 0.82;
      ctx.beginPath();
      ctx.moveTo(a.x, a.y);
      ctx.lineTo(b.x, b.y);
      ctx.lineTo(c.x, c.y);
      ctx.lineTo(d.x, d.y);
      ctx.closePath();
      ctx.fill();

      ctx.globalAlpha = 0.34;
      ctx.strokeStyle = "rgba(220,255,255,.45)";
      ctx.lineWidth = 0.45;
      ctx.stroke();
    }
  }
  ctx.restore();

  // Neon contour lines along rows
  ctx.save();
  ctx.lineWidth = 1.2;
  ctx.shadowColor = C.cyan;
  ctx.shadowBlur = glow;
  for (let j = 0; j < rows.length; j += 2) {
    ctx.strokeStyle = j % 4 === 0 ? "rgba(100,245,255,.58)" : "rgba(255,255,255,.28)";
    ctx.beginPath();
    rows[j].forEach((p, i) => {
      const q = project(panel, p);
      if (i === 0) ctx.moveTo(q.x, q.y);
      else ctx.lineTo(q.x, q.y);
    });
    ctx.stroke();
  }

  // Moving scan ridge
  const scanY = (phase * 5.0) % 24;
  ctx.strokeStyle = "rgba(255,245,90,.94)";
  ctx.shadowColor = "#fff042";
  ctx.shadowBlur = glow + 8;
  ctx.lineWidth = 2.2;
  ctx.beginPath();
  for (let x = 0; x <= 50; x += 1) {
    const p = {x, y:scanY, z:ampFn(x, scanY, phase, ripple)};
    const q = project(panel, p);
    if (x === 0) ctx.moveTo(q.x, q.y);
    else ctx.lineTo(q.x, q.y);
  }
  ctx.stroke();

  // Energy points
  for (let k = 0; k < 12; k++) {
    const x = (k * 7 + phase * 11) % 50;
    const y = (k * 5 + phase * 8) % 24;
    const z = ampFn(x, y, phase, ripple);
    const q = project(panel, {x,y,z});
    ctx.fillStyle = "#fff785";
    ctx.shadowColor = "#fff042";
    ctx.shadowBlur = glow + 5;
    ctx.globalAlpha = .75;
    ctx.beginPath();
    ctx.arc(q.x, q.y, 1.8 + 1.2 * Math.sin(phase * 3 + k), 0, Math.PI * 2);
    ctx.fill();
  }
  ctx.restore();
}

function drawColorbar(panel) {
  const barX = panel.cx + panel.sx * 0.72;
  const barY = panel.cy - panel.sz * 0.58;
  const barW = 18;
  const barH = panel.sz * 1.15;

  const grad = ctx.createLinearGradient(0, barY + barH, 0, barY);
  grad.addColorStop(0, C.purple);
  grad.addColorStop(.28, C.blue);
  grad.addColorStop(.52, C.teal);
  grad.addColorStop(.76, C.green);
  grad.addColorStop(1, C.yellow);

  ctx.save();
  ctx.fillStyle = grad;
  ctx.shadowColor = "rgba(255,240,90,.65)";
  ctx.shadowBlur = 13;
  ctx.fillRect(barX, barY, barW, barH);
  ctx.strokeStyle = "rgba(240,250,255,.68)";
  ctx.lineWidth = 1;
  ctx.strokeRect(barX, barY, barW, barH);

  panel.zTicks.forEach(z => {
    const t = (z - panel.zmin) / (panel.zmax - panel.zmin);
    const y = barY + barH - t * barH;
    ctx.strokeStyle = "rgba(255,255,255,.75)";
    ctx.beginPath();
    ctx.moveTo(barX + barW, y);
    ctx.lineTo(barX + barW + 7, y);
    ctx.stroke();
    text(z.toFixed(2), barX + barW + 12, y, 11, "left");
  });

  text("Amplitude", barX + barW / 2, barY + barH + 34, 13, "center", 0, 760);
  ctx.restore();
}

function drawBackground(phase) {
  ctx.clearRect(0, 0, W, H);

  // Soft star/particle field
  ctx.save();
  ctx.globalAlpha = 0.18;
  for (let i = 0; i < 70; i++) {
    const x = (i * 157 + phase * 25) % W;
    const y = (i * 83 + Math.sin(phase + i) * 8) % H;
    ctx.fillStyle = "white";
    ctx.fillRect(x, y, 1.1, 1.1);
  }
  ctx.restore();

  // Orbital streaks
  ctx.save();
  ctx.strokeStyle = "rgba(56,175,255,.16)";
  ctx.lineWidth = 1.3;
  for (let r = 0; r < 6; r++) {
    ctx.beginPath();
    const cy = H * (0.28 + r * 0.12);
    for (let i = 0; i <= 100; i++) {
      const a = (i / 100) * Math.PI * 1.6 + phase * 0.22 + r * 0.4;
      const x = W * .50 + Math.cos(a) * (W * (.42 + r * .025));
      const y = cy + Math.sin(a) * (H * .10);
      if (i === 0) ctx.moveTo(x,y);
      else ctx.lineTo(x,y);
    }
    ctx.stroke();
  }
  ctx.restore();
}

function render(now) {
  const speed = Number(speedSlider.value);
  const ripple = Number(rippleSlider.value);
  const glow = Number(glowSlider.value);
  const phase = phaseHold + ((now - start) / 1000) * speed;

  frame++;
  drawBackground(phase);

  const top = {
    cx: W * 0.42, cy: H * 0.29,
    sx: Math.min(W * 0.58, 770),
    sy: Math.min(W * 0.30, 390),
    sz: Math.min(H * 0.22, 210),
    zmin: 0.48, zmax: 0.64,
    zTicks: [0.48,0.50,0.52,0.54,0.56,0.58,0.60,0.62,0.64]
  };

  const bottom = {
    cx: W * 0.42, cy: H * 0.70,
    sx: Math.min(W * 0.58, 770),
    sy: Math.min(W * 0.30, 390),
    sz: Math.min(H * 0.22, 210),
    zmin: 0.24, zmax: 0.32,
    zTicks: [0.24,0.25,0.26,0.27,0.28,0.29,0.30,0.31,0.32]
  };

  drawGrid(top);
  drawSurface(top, ampTop, phase, ripple, glow);
  drawAxes(top);
  drawColorbar(top);

  drawGrid(bottom);
  drawSurface(bottom, ampBottom, phase + 0.65, ripple, glow);
  drawAxes(bottom);
  drawColorbar(bottom);

  const sample = (phase * 8) % 50;
  const tHour = (phase * 3.7) % 24;
  const liveAmp = ampTop(sample, tHour, phase, ripple);

  frameReadout.textContent = `Frame: ${frame}`;
  waveReadout.textContent = `Surface phase: ${phase.toFixed(2)}`;
  sampleReadout.textContent = `Current sample: ${sample.toFixed(1)}/hr`;
  ampReadout.textContent = `Amplitude: ${liveAmp.toFixed(3)}`;

  if (running) requestAnimationFrame(render);
}

requestAnimationFrame(render);

pauseBtn.addEventListener("click", () => {
  running = !running;
  if (running) {
    pauseBtn.textContent = "Pause";
    start = performance.now();
    requestAnimationFrame(render);
  } else {
    phaseHold += ((performance.now() - start) / 1000) * Number(speedSlider.value);
    pauseBtn.textContent = "Resume";
  }
});

resetBtn.addEventListener("click", () => {
  phaseHold = 0;
  frame = 0;
  start = performance.now();
  if (!running) {
    running = true;
    pauseBtn.textContent = "Pause";
    requestAnimationFrame(render);
  }
});
</script>
</body>
</html>
"""

HTML(html_content)